# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Convert metadata to dictionary if needed
metadata = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', '')}: {getattr(dataset.metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their `@id`s
record_sets = list(dataset.metadata.record_sets) if hasattr(dataset.metadata, 'record_sets') else []
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', None)}\n  @id: {getattr(rs, '@id', None)}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    if hasattr(rs, 'fields'):
        print("  Fields:")
        for field in rs.fields:
            print(f"    - name: {getattr(field, 'name', None)}, @id: {getattr(field, '@id', None)}")
    print()

# If no record sets found, print instructions
if not record_sets:
    print("No record sets found in the metadata. Check the schema for 'record_set' or similar elements.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# When the Croissant schema is complex, record sets/fields must be referenced by their `@id` field.
# We'll extract all available record sets into Pandas DataFrames.
dataframes = {}
record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    if rs_id is not None:
        print(f"Loading records for RecordSet: {rs.name} (@id={rs_id})...")
        try:
            records = list(dataset.records(record_set=rs_id))
            df = pd.DataFrame(records) if records else pd.DataFrame()
            dataframes[rs_id] = df
            record_set_ids.append(rs_id)
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"  Could not load records for {rs_id}: {e}")
    else:
        print("  No @id found, skipping record set.")

# If dataframes are loaded, show the first record set DataFrame
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nExample DataFrame for RecordSet '@id': {main_rs_id}")
    display(dataframes[main_rs_id].head())
else:
    print("No record sets could be loaded as DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section will demonstrate filtering, normalization, and grouping for one loaded record set.

In [ ]:
# Example: Choose a record set and a numeric field (by `@id`) for analysis

if record_set_ids:
    rs_id = main_rs_id
    df = dataframes[rs_id]
    print(f"Columns for RecordSet {rs_id}: {df.columns.tolist()}")

    # AUTO-DETECTION or replace with the exact `@id` of a numeric field, e.g. 'cr:mw' for molecular weight
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64']]
    if len(possible_numeric_fields) == 0:
        # Try to auto-convert columns to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                continue
        possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int, float, 'int64', 'float64']]
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}\n")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
        # Filter
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping (by a categorical field, if exists)
        possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id} (showing mean of numerics):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for this record set.\n")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and possible_numeric_fields:
    # Plot the distribution of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()
    # If grouped_df is available, show a bar plot of group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,4))
        grouped_df[numeric_field_id].plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We explored the FAIR² dataset on mass spectrometry analysis of extracellular vesicles using the `mlcroissant` library.
- The dataset metadata, available record sets, and field information can be extracted programmatically and referenced by their `@id` as shown.
- We demonstrated filtering, normalization, grouping, and basic visualization techniques for further analysis.
- For deeper insights, consult the dataset documentation and expand EDA by exploring additional record sets and complex relationships.